## publish 

> This is the `publish` module for the ERA5 dataset pipeline. It defines a functions that make use of the `pyDataverse` library and API to publish our outputs to the Harvard Dataverse.

In [2]:
#| default_exp publish

In [3]:
#| hide
from nbdev.showdoc import *

First, we'll test out the API by pinging the Harvard DataVerse


In [4]:
#| export
#| 
import hydra
import yaml
import json
from tqdm import tqdm
from pyprojroot import here

In [5]:
api_token_file = here() / "sandbox/dataverse_api_key.yml"
with open(api_token_file, "r") as f:
    config = yaml.load(f, Loader=yaml.BaseLoader)

Now, following the [docs]() for the dataverse tutorial, load a NativeAPI up:

In [ ]:
#| export
from pyDataverse.api import NativeApi

The NativeAPI is a catchall API object to be able to do general stuff:


In [10]:
api = NativeApi(config['base_url'], config['api_token'])
resp=api.get_info_version()
#resp.text()

In [11]:
resp.json()

{'status': 'OK', 'data': {'version': '6.6', 'build': 'iqss-4'}}

Looks good! Now that we know that it works, we can think more
about how to publish data there.

## Harvard Dataverse

Let's create a dummy dataset with the components we're
planning to upload, and then upload and promptly delete it.

To do that, we must import the `models` module and create a Dataset object:


In [12]:
from pyDataverse.models import Dataset
ds = Dataset()

This `ds` object is pretty straightforward since it doesn't contain anything yet:

In [13]:
ds.get()

{}

We can populate the object from the dummy data on the github repo:


In [14]:
from pyDataverse.utils import read_file
from urllib.request import urlretrieve
import tempfile

url = "https://raw.githubusercontent.com/gdcc/pyDataverse/refs/heads/main/tests/data/user-guide/dataset.json"

with tempfile.NamedTemporaryFile(mode='w+') as tmp:
    urlretrieve(url, tmp.name)
    ds.from_json(read_file(tmp.name))

We have to validate the JSON correctly:


In [15]:
ds.validate_json()


True

Modifying it is easy:


In [16]:
ds.set({"title": "Youth from Austria 2005"})
ds.get()

{'citation_displayName': 'Citation Metadata',
 'title': 'Youth from Austria 2005',
 'author': [{'authorName': 'LastAuthor1, FirstAuthor1',
   'authorAffiliation': 'AuthorAffiliation1'}],
 'datasetContact': [{'datasetContactEmail': 'ContactEmail1@mailinator.com',
   'datasetContactName': 'LastContact1, FirstContact1'}],
 'dsDescription': [{'dsDescriptionValue': 'DescriptionText'}],
 'subject': ['Medicine, Health and Life Sciences']}

Now, to create the dataset we use the API:


In [ ]:
#| eval: false
#| this is only run in interactive sessions for demo purposes
resp = api.create_dataset(":root", ds.json())

If you caught the `resp` object, it contains the PID for the newly created dataset.

However, if you didn't you can use the SearchAPI to find it:


In [ ]:
#| export
from pyDataverse.api import SearchApi

In [ ]:
from pyDataverse.api import SearchApi

search_api = SearchApi(config['base_url'], config['api_token'])
resp = search_api.search("Youth from Austria", data_type="dataset")
results = resp.json()['data']['items']
result = [x for x in results if "Youth from Austria" in x['name']][0]
result

In [ ]:
pid = result['global_id']

Now to look at the data we created using the NativeAPI again, and delete the dataset:


In [ ]:
uploaded_ds = api.get_dataset(pid)
uploaded_ds.json()['data']

resp = api.delete_dataset(pid)
resp.json()

With that understanding, we can develop a quick module to do the following:

1. Make the dataset LEGO Compatible
2. Upload and publish the data to dataverse

## LEGO Compatibility

Let's take an example file to use as a model for LEGO compatibility

In [17]:
#| export
import geopandas as gpd
import pandas as pd
import re
import glob

In [18]:
ex = gpd.read_parquet(here() / "data" / "testing" / "madagascar_environmental_exposure-era5_healthshed_2m_dewpoint_temperature_2009_6.parquet")
ex.describe()

,day_01_daily_mean,day_02_daily_mean,day_03_daily_mean,day_04_daily_mean,day_05_daily_mean,day_06_daily_mean,day_07_daily_mean,day_08_daily_mean,day_09_daily_mean,day_10_daily_mean,...,day_21_daily_max,day_22_daily_max,day_23_daily_max,day_24_daily_max,day_25_daily_max,day_26_daily_max,day_27_daily_max,day_28_daily_max,day_29_daily_max,day_30_daily_max
count,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,...,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000,2766.000000
mean,290.101105,290.251129,290.299927,290.669952,290.294189,289.835541,289.162964,288.179321,287.109406,287.835236,...,288.465210,289.768005,290.134491,290.183838,289.658630,288.893921,288.319275,287.971619,287.961121,284.683014
std,3.746835,3.516243,3.244272,2.700433,2.922641,3.155868,2.765681,3.065981,3.427631,2.879560,...,3.016459,3.004185,2.842475,3.137010,3.071827,3.578434,3.335460,3.472515,4.297029,4.488029
min,282.279327,283.185089,283.726196,284.917725,284.198273,282.932312,283.175507,281.274048,279.423431,282.543793,...,281.077148,282.664795,283.084229,283.270264,283.417969,282.003174,280.433594,281.191406,280.080566,277.250977
25%,286.597130,287.106430,287.825439,288.353699,287.752289,287.107872,286.748573,285.718079,284.398026,285.039490,...,286.215302,287.150391,287.687744,287.582214,287.120544,285.636597,285.718857,284.739990,284.020073,281.114044
50%,290.789474,290.788528,290.232422,290.638428,290.559021,290.241394,289.157364,288.221802,287.098816,288.132477,...,288.559937,290.283844,290.668625,290.602966,289.662033,289.353104,287.772232,288.051758,288.368774,283.901611
75%,293.350601,293.341553,293.279350,293.184204,292.985542,292.658203,291.717346,290.950699,290.250725,290.077141,...,291.194336,292.494934,292.651917,293.070877,292.559906,292.236084,291.029053,290.982109,291.481934,288.649673
max,295.952759,296.164429,296.283569,296.434967,296.162384,296.076172,295.498718,294.170776,293.808472,293.877411,...,294.056519,294.639008,295.159424,295.899170,295.123169,295.179199,295.344360,294.466919,295.652588,294.524658


We know that the LEGO data model should look like this:

```
<main lab folder>/lego
├── <domain>
│   ├── <subdomain>__<data_source>
│   │   ├── <geo_resolution>__<time_resolution>
│   │   │   ├── <filename>_yyyy.parquet
```

So, for the above file, we'll end up with the LEGO path `data/environmental/exposures_era5/healthshed_monthly/dewpoint_2024.parquet`. In it, we should have the following columns:


```
healthshed_id  year month day stat_1 stat_2 ... stat_n   
```


This means we should read in all of the exposures for a single timepoint at once. 
I think the smart thing to do is use a glob string to gather all of the pertinent files.
This will be the first function we export to the library:


In [ ]:
#| export
# 

def gather_exposure_geodataframes(
    glob_string: str,   # string for the path to search for the pertinent files
    polygon_id: str,    # the string signifying the healthshed ID of the polygon
    exposure: str       # the exposure name
    )-> list:
    "Read in a list of geo dataframes from the same time frame and merge them"

    # first get the initial one so we have the polygon ID and geometry
    frames = glob.glob(str(glob_string))
    initial_gdf=gpd.read_parquet(frames[0])
    merged_df = []
  
    for f in tqdm(frames, desc="Processing files"):
        # read in as a regular dataframe by ignoring geometry
        df = gpd.read_parquet(f).drop(["geometry"], axis=1) 
        
        # get the year and month
        # Extract year and month
        search_str = rf'_{exposure}_(\d{{4}})_(\d{{1,2}})\.parquet$'
        match = re.search(search_str, f)

        if match:
            year = int(match.group(1))
            month = int(match.group(2))
            #print(f"Year: {year}, Month: {month}")
        else:
            raise ValueError(f"Could not extract year and month from filename: {search_str} {f}")
            
        df['exposure'] = exposure
        df['month'] = month
        df['year'] = year

        # Step 1: Melt all day columns (leave 'month' and 'year' as identifiers)
        df_long = df.melt(id_vars=[polygon_id, "exposure", "year", "month"], var_name="day_stat", value_name="value")

        # Step 2: Extract day and stat type from column names
        # Example column: "day_01_daily_mean"
        df_long[["day", "stat"]] = df_long["day_stat"].str.extract(r"day_(\d{2})_daily_(mean|max|min|total)")

        # Optional: convert 'day' and month to integer
        df_long["day"] = df_long["day"].astype(int)
        df_long["month"] = df_long["month"].astype(int)

        # Drop the original combined column
        df_long = df_long.drop(columns="day_stat")

        # Reorder columns
        df_long = df_long[[polygon_id, "exposure", "year", "month", "day", "stat", "value"]]

        df_long = df_long.sort_values(by=["year", "month", "day"])
        df_clean = df_long.pivot(index=[polygon_id, "exposure", "year", "month", "day"], columns="stat", values="value").reset_index()
        merged_df.append(df_clean)

    return [pd.concat(merged_df).reset_index(drop=True), initial_gdf[[polygon_id, "geometry"]]]

In [38]:
frames = here() / "data" / "testing" / "*madagascar*"

merged = gather_exposure_geodataframes(frames, "fs_uid", "2m_dewpoint_temperature")
merged[0].describe()

Processing files:   0%|          | 0/3 [00:00<?, ?it/s]

Processing files: 100%|██████████| 3/3 [00:01<00:00,  1.53it/s]


stat,year,month,day,max,mean,min
count,254472.0,254472.000000,254472.000000,254472.000000,254472.000000,254472.000000
mean,2009.0,5.663043,15.836957,292.115845,290.383850,288.571930
std,0.0,3.701585,8.854244,3.794787,4.128042,4.721353
min,2009.0,1.000000,1.000000,277.250977,273.298462,268.284668
25%,2009.0,1.000000,8.000000,289.436615,287.414513,285.107178
50%,2009.0,6.000000,16.000000,292.382812,290.696609,288.960571
75%,2009.0,10.000000,23.250000,294.812210,293.349281,292.088867
max,2009.0,10.000000,31.000000,300.528076,299.109772,298.311462


This returns one file with all of the geometries and one file
with the statistics and exposures.

Now, with this, we can move on. The dataset was created in the UI and is available via search and test out how to upload it:


In [ ]:
resp = search_api.search("ERA5", data_type="dataset")

results = resp.json()['data']['items']

result = [x for x in results if "ERA5" in x['name']][0]
era5_pid = result['global_id']
result

In [ ]:
#| export

from pyDataverse.models import Datafile
import os
import pathlib

We'll upload directly from file. In the case of ERA5 vs. LEGO, we
store the file on disk as LEGO hierarchy, but to upload it to dataverse
using a flat filename (since creating subdatasets to represent directories might be 
a bit of a hassle)

In [ ]:
# assuming the file has a path on disk like:
f_out = "environmental/exposures_era5/healthshed_daily/dewpoint_2024.parquet"
os.makedirs(here() / "data" / "testing" / os.path.dirname(f_out), exist_ok=True)
aggregations, geo = merged
aggregations.to_parquet(here() / "data" / "testing" / f_out, index=False)

datafile = Datafile()
datafile.set({
    # the id of the era5 dataset 
    "pid": era5_pid,
    # the path to the file on disk goes here
    "filename": str(here() / "data" / "testing" / f_out),
    # use the "label" to name the file
    "label": f_out.replace("/", "-")
})

In [ ]:
#| eval: false
resp = api.upload_datafile(era5_pid, str(here() / "data" / "testing" / f_out), datafile.json())

Pretty simple!

Now, we just need a main function to upload this data. The final upload is one file per
exposure per year, so these should be the variables we gather data for.

We should get some functionality to gather the groups of these files automatically, based on
the hydra config:

In [ ]:
#| export
from hydra import initialize, compose
from omegaconf import OmegaConf, DictConfig
from tqdm import tqdm

In [ ]:
target_dir = here() / "data" / "intermediate"

with initialize(version_base=None, config_path="../conf"):
    cfg = compose(config_name='config.yaml')

cfg.development_mode = False
#cfg.query['year'] = 2017
#cfg.query['month'] = 11
#cfg.query['geography'] = "nepal"

In [ ]:
#| export

@hydra.main(version_base=None, config_path="../../conf", config_name="config")
def main(cfg: DictConfig) -> None:

    variables_dict = {
        "2m_temperature": "t2m",
        "2m_dewpoint_temperature": "d2m",
        "volumetric_soil_water_layer_1": "swvl1",
        "total_precipitation": "tp"
    }

    print(OmegaConf.to_yaml(cfg))

    #prep dataverse
    api_token_file = here() / "sandbox/dataverse_api_key.yml"
    with open(api_token_file, "r") as f:
        apiconfig = yaml.load(f, Loader=yaml.BaseLoader)
    api = NativeApi(apiconfig['base_url'], apiconfig['api_token'])
    search_api = SearchApi(apiconfig['base_url'], apiconfig['api_token'])
    resp = search_api.search("ERA5", data_type="dataset")

    results = resp.json()['data']['items']

    result = [x for x in results if "ERA5" in x['name']][0]
    era5_pid = result['global_id']

    for geography in cfg.geographies:
        for year in cfg.query['year']:
            for variable, v in variables_dict.items():
                
                print(f"Processing {geography} for {variable} in {year}")
                glob_string = here() / "data" / "intermediate" / f"*{geography}*{variable}*{year}*"
                print(f"Glob: {glob_string}")
                polygon_id = cfg.geographies[geography]['unique_id']
                print(f"polygon_id: {polygon_id}")
                merged = gather_exposure_geodataframes(glob_string, polygon_id, variable)
                print(merged[0].head())
                print(merged[1].head())

                output_dir = here() / "data" / "output" 
                
                f_out = f"environmental/exposures_era5/healthshed_daily/{geography}_{v}_{year}.parquet"
                os.makedirs(output_dir / os.path.dirname(f_out), exist_ok=True)
                output_path = output_dir / f_out

                print(f"Writing to {output_path}")
                merged[0].to_parquet(output_path, index=False)
                

                print(f"Uploading {f_out.replace('/', '-')} to Dataverse...")
                # upload to dataverse
                datafile = Datafile()
                datafile.set({
                    "pid": era5_pid,
                    "filename": str(output_path),
                    "label": f_out.replace("/", "-")
                })

                resp = api.upload_datafile(era5_pid, output_path, datafile.json())
                assert resp.json()['status'] == "OK", f"Failed to upload datafile: {resp.text}"
        
        # also save the geometry for the region 
        merged[1].to_parquet(output_path.parent / f"{geography}_geometry.parquet", index=False)

        # and upload it to dataverse
        datafile = Datafile()
        datafile.set({
            "pid": era5_pid,
            "filename": str(output_path.parent / f"{geography}_geometry.parquet"),
            "label": f"{geography}_geometry.parquet"
        })

        resp = api.upload_datafile(era5_pid, output_path.parent / f"{geography}_geometry.parquet", datafile.json())
        assert resp.json()['status'] == "OK", f"Failed to upload geometry datafile: {resp.text}"

    print("All files processed and uploaded successfully.")
            

In [ ]:
#| export
#| eval: false
try: from nbdev.imports import IN_NOTEBOOK
except: IN_NOTEBOOK=False

if __name__ == "__main__" and not IN_NOTEBOOK:
    main()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()